# 판다스 총 정리
이번 파일에서는 지금까지 봐온 pandas의 내용들을 한번 처음부터 정리해보려 합니다.

한번에 모두 다루기에는 방대한 양의 메서드 및 기능, 옵션 등을 지원하기 때문에 어느정도의 생략이 존재할 수 있습니다.

우선 Pandas의 진행 흐름은 아래와 같으면 좋겠다 판단하였습니다.
1. 판다스 기본 데이터 타입 등
2. 판다스 기본 데이터 형태 확인
3. DataFrame와 Series의 관계
4. DataFrame 기본 정보 확인
5. 결측치 처리
6. 인덱싱과 기본 통계
7. concat와 merge를 이용한 데이터 결합

> 사실 이것저것 다루려 생각한 옵션이나 메서드가 많았는데 생각보다 너무 많고 글을 쓰는데 1시간이 걸리다보니 뭔가 지치더라고요... 그래서 그냥 어느정도 생략이 더 생겼습니다...

# DataFrame
Pandas의 `DataFrame`는 크게 3가지 요소로 이루어져 있습니다.
- Columns: 열 (feature)
- Index: 행 (인덱스, 식별자 등)
- values: 값 (ndarray 기준 [Columns, Index]에 존재하는 모든 값들)

이때 values의 타입은 ndarray이며
Columns와 Index의 타입은 Index라는 별도의 타입으로 이 또한 ndarrray로 이루어져 있습니다. (인덱스의 경우에는 RangeIndex로 이루어지기도 합니다.)

참고로 Columns은 불변 시퀀스이기 때문에 하나의 값만 바꿀 순 없습니다. `.rename({"old": "new"})`를 써도 새로 만들어 갈아끼게 됩니다.

In [62]:
import pandas as pd, numpy as np

df = pd.DataFrame({
    "name": ["kim", "lee", "park"],
    "age": [20, 25, 30],
    "score": [80, 90, 85]
})

print("\n --- columns --- \n")
print(df.columns)
print(type(df.columns))
print(type(df.columns.values))
print(df.columns.dtype)


print("\n --- range index --- \n")
print(df.index)
print(type(df.index))
print(type(df.index.values))
print(df.index.dtype)

df.index = [0, 1, 2]
print("\n --- index --- \n")
print(df.index)
print(type(df.index))
print(type(df.index.values))
print(df.index.dtype)

print("\n --- values --- \n")
print(df.values)
print(type(df.values))


 --- columns --- 

Index(['name', 'age', 'score'], dtype='str')
<class 'pandas.Index'>
<class 'pandas.arrays.StringArray'>
str

 --- range index --- 

RangeIndex(start=0, stop=3, step=1)
<class 'pandas.RangeIndex'>
<class 'numpy.ndarray'>
int64

 --- index --- 

Index([0, 1, 2], dtype='int64')
<class 'pandas.Index'>
<class 'numpy.ndarray'>
int64

 --- values --- 

[['kim' 20 80]
 ['lee' 25 90]
 ['park' 30 85]]
<class 'numpy.ndarray'>


# DataFrame = Series 묶음
DataFrame의 경우에는 여러개의 Series가 붙는 형태가 됩니다.

In [63]:
for c in df.columns:
    print(f"{c}: {type(df[c])}")
print(f'df[["age", "name"]]: {type(df[["age", "name"]])}')

name: <class 'pandas.Series'>
age: <class 'pandas.Series'>
score: <class 'pandas.Series'>
df[["age", "name"]]: <class 'pandas.DataFrame'>


# DataFrame 기본 정보를 보는 법
DataFrame는 매우 많이 사용되는 데이터 타입인 만큼 들어있는 많은 데이터를 요약하여 확인이 가능합니다. 

기능은 아래와 같습니다.

In [64]:
# 데이터 형태 열별 요약
print("\n --- df.info() --- \n")
print(df.info())

# 데이터 통계 요약 (str(obj)와 number을 나누어 판별)
print("\n  --- df.describe(include='all') --- \n")
print(df.describe(include="all"))

print("\n  --- df.describe(include='number') --- \n")
print(df.describe(include="number"))

print("\n  --- df.describe(include='str(object)') --- \n")
print(df.describe(include="str"))

# 데이터 위쪽 5개
print("\n --- df.head(5) --- \n")
print(df.head(5))

# 데이터 아래쪽 5개
print("\n --- df.tail(5) --- \n")
print(df.tail(5))


# 기타 메타데이터
print("\n --- metadata --- \n")
print(f"df.shape: {df.shape}")
print(f"df.dtypes: \n{df.dtypes}")


 --- df.info() --- 

<class 'pandas.DataFrame'>
Index: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    3 non-null      str  
 1   age     3 non-null      int64
 2   score   3 non-null      int64
dtypes: int64(2), str(1)
memory usage: 96.0 bytes
None

  --- df.describe(include='all') --- 

       name   age  score
count     3   3.0    3.0
unique    3   NaN    NaN
top     kim   NaN    NaN
freq      1   NaN    NaN
mean    NaN  25.0   85.0
std     NaN   5.0    5.0
min     NaN  20.0   80.0
25%     NaN  22.5   82.5
50%     NaN  25.0   85.0
75%     NaN  27.5   87.5
max     NaN  30.0   90.0

  --- df.describe(include='number') --- 

        age  score
count   3.0    3.0
mean   25.0   85.0
std     5.0    5.0
min    20.0   80.0
25%    22.5   82.5
50%    25.0   85.0
75%    27.5   87.5
max    30.0   90.0

  --- df.describe(include='str(object)') --- 

       name
count     3
unique    3
top     kim
freq      1

 

# 결측치
결측치에 대한 메서드들 입니다.

In [ ]:
# 기본 결측치 메서드들 
df.iloc[0, 0] = pd.NA
df.iloc[1, 2] = pd.NA


# 결측치 DataFrame boolean으로 반환
print("\n --- df.isna() --- \n")
print(f"{df.isna()}")

# 열별로 더하기
print("\n --- df.isna().sum() --- \n")
print(df.isna().sum())

# 결측치 행 제거
print("\n --- df.dropna(axis=0) 0 is default --- \n")
print(df.dropna())

# 특정 열에 대해서 결측치 행 제거
print("\n --- df.dropna(subset = ['name'], axis=0) --- \n")
print(df.dropna(subset = ['name'], axis=0))



 --- df.isna() --- 

    name    age  score
0   True  False  False
1  False  False   True
2  False  False  False

 --- df.isna().sum() --- 

name     1
age      0
score    1
dtype: int64

 --- df.dropna(axis=0) 0 is default --- 

   name  age  score
2  park   30   85.0

 --- df.dropna(subset = ['name'], axis=0) 0 is default --- 

   name  age  score
1   lee   25    NaN
2  park   30   85.0


In [66]:
# 고유값 확인
print("\n --- new dataset ---\n")
df = pd.DataFrame({
    "name": ["a", "b" ,"c"],
    "age": [12, 11, 12]
})
print(df)

print("\n --- df.value_counts() ---\n")
print(df.value_counts())


 --- new dataset ---

  name  age
0    a   12
1    b   11
2    c   12

 --- df.value_counts() ---

name  age
a     12     1
b     11     1
c     12     1
Name: count, dtype: int64


# 인덱싱
인덱싱은 뭔가 DataFrame의 꽃이라고 부를 수 있으며 해당 방법은 굉장히 많습니다.


In [67]:
print("\n --- new dataset --- \n")
df = pd.DataFrame({
    "name": ["철수", "영희", "민수", "지은", "현우", "서연"],
    "age": [23, 25, np.nan, 22, 27, 24],
    "score": [85, np.nan, 78, 92, 88, np.nan],
    "city": ["서울", "부산", "대구", np.nan, "인천", "광주"]
})

print(df)


 --- new dataset --- 

  name   age  score city
0   철수  23.0   85.0   서울
1   영희  25.0    NaN   부산
2   민수   NaN   78.0   대구
3   지은  22.0   92.0  NaN
4   현우  27.0   88.0   인천
5   서연  24.0    NaN   광주


In [74]:
print("##### 기본 형태 #####")
print("\n --- df.sum() --- ")
print(df.sum())
print("\n --- df.var(numeric_only=True) DataFrame의 var ddof가 1입니다. (표본분산 [/(n-1)]) --- ")
print(df.var(numeric_only=True))
print("\n --- df.std(numeric_only=True) DataFrame의 std는 ddof가 1입니다. (표본표준편차 [/(n-1)]) --- ")
print(df.std(numeric_only=True))


##### 기본 형태 #####

 --- df.sum() --- 
name     철수영희민수지은현우서연
age             121.0
score           343.0
city       서울부산대구인천광주
dtype: object

 --- df.var(numeric_only=True) DataFrame의 var ddof가 1입니다. (표본분산 [/(n-1)]) --- 
age       3.700000
score    34.916667
dtype: float64

 --- df.std(numeric_only=True) DataFrame의 std는 ddof가 1입니다. (표본표준편차 [/(n-1)]) --- 
age      1.923538
score    5.909033
dtype: float64


# concat와 merge
인덱스끼리의 결합에서는 특수하게 작용하는 메서드들이 많습니다.

**concat**는 두 DataFrame를 세로 또는 가로로 합치는 메서드이고

**merge**는 SQL의 `join`과 동일한 작업을 수행해줍니다. 방식은 `how`네임드 인자에 `inner`, `left`, `right`, `outer`이 가능합니다. 같은 열이 있으면 자동으로 각각의 열에 `_x`와 `_y`가 붙으며 `suffixes`로 조정 가능합니다. (`df.join(df)`는 `lsuffix`, `rsuffix`를 따로 지정합니다.)

여러 작용들을 알아보겠습니다.

In [79]:
# concat(axis=0): 세로로 붙이기
df1 = pd.DataFrame({
    "key": ["A", "B"],
    "value": [10, 20]
})

df2 = pd.DataFrame({
    "key": ["C", "D"],
    "value": [30, 40]
})

print("\n --- df1 --- \n")
print(df1)
print("\n --- df2 --- \n")
print(df2)

print("\n --- pd.concat([df1, df2], axis=0), 기존 index 유지 --- \n")
print(pd.concat([df1, df2]))

print("\n --- pd.concat([df1, df2], axis=0, ignore_index=True) 인덱스가 0부터 새로 쌓입니다. --- \n")
print(pd.concat([df1, df2], ignore_index=True))


 --- df1 --- 

  key  value
0   A     10
1   B     20

 --- df2 --- 

  key  value
0   C     30
1   D     40

 --- pd.concat([df1, df2], axis=0), 기존 index 유지 --- 

  key  value
0   A     10
1   B     20
0   C     30
1   D     40

 --- pd.concat([df1, df2], axis=0, ignore_index=True) 인덱스가 0부터 새로 쌓입니다. --- 

  key  value
0   A     10
1   B     20
2   C     30
3   D     40


In [81]:
# concat(axis=1): 가로로 붙이기
df1 = pd.DataFrame({
    "key": ["A", "B", "C"]
}, index=[0, 1, 2])

df2 = pd.DataFrame({
    "value": [10, 20, 30]
}, index=[1, 2, 3])

print("\n --- df1 --- \n")
print(df1)
print("\n --- df2 --- \n")
print(df2)

print("\n --- pd.concat([df1, df2], axis=1), index 기준으로 붙음 --- \n")
print(pd.concat([df1, df2], axis=1))

print("\n --- df.reset_index() 후 concat() --- \n")
print(pd.concat([
    df1.reset_index(drop=True),
    df2.reset_index(drop=True)
], axis=1))


 --- df1 --- 

  key
0   A
1   B
2   C

 --- df2 --- 

   value
1     10
2     20
3     30

 --- pd.concat([df1, df2], axis=1), index 기준으로 붙음 --- 

   key  value
0    A    NaN
1    B   10.0
2    C   20.0
3  NaN   30.0

 --- df.reset_index() 후 concat() --- 

  key  value
0   A     10
1   B     20
2   C     30


In [ ]:
# merge: key 컬럼 기준으로 합치기
df1 = pd.DataFrame({
    "key": ["A", "B", "C"],
    "value1": [10, 20, 30]
}, index=[1,2,3])

df2 = pd.DataFrame({
    "key": ["B", "C", "A"],
    "value2": [200, 300, 100]
}, index=[3,5,6])

print("\n --- df1 --- \n")
print(df1)
print("\n --- df2 --- \n")
print(df2)

print("\n --- pd.merge(df1, df2, on='key') right_on, left_on의 결합니다.index는 새로 만들어집니다. --- \n")
print(pd.merge(df1, df2, on="key"))




 --- df1 --- 

  key  value1
1   A      10
2   B      20
3   C      30

 --- df2 --- 

  key  value2
3   B     200
5   C     300
6   A     100

 --- merge(on='key') right_on, left_on의 결합니다.index는 새로 만들어집니다. --- 

  key  value1  value2
0   A      10     100
1   B      20     200
2   C      30     300


In [ ]:
# merge: index 기준으로 합치기
df1 = pd.DataFrame({
    "value1": [10, 20, 30]
}, index=[101, 102, 103])

df2 = pd.DataFrame({
    "value2": [100, 200, 300]
}, index=[101, 102, 103])

print("\n --- df1 --- \n")
print(df1)
print("\n --- df2 --- \n")
print(df2)

print("\n --- pd.merge(df1, df2, left_index=True, right_index=True) merge index 기준 --- \n")
print(pd.merge(df1, df2, left_index=True, right_index=True))

print("\n --- df1.join(df2) merge by index와 동일합니다. --- \n")
print(df1.join(df2))


 --- df1 --- 

     value1
101      10
102      20
103      30

 --- df2 --- 

     value2
101     100
102     200
103     300


In [87]:

# merge: key가 중복되면 행이 늘어날 수 있음
df1 = pd.DataFrame({
    "key": ["A", "B"],
    "value1": [10, 20]
})

df2 = pd.DataFrame({
    "key": ["A", "A", "B"],
    "type": ["x", "y", "z"],
    "value2": [100, 150, 200]
})

print("\n --- df1 --- \n")
print(df1)
print("\n --- df2 --- \n")
print(df2)

print("\n --- (pd.merge(df1, df2, on='key') key 중복 시 행이 늘어남 --- \n")
print(pd.merge(df1, df2, on="key"))

print("\n --- 중복 key 확인 --- \n")
print(df2[df2["key"].duplicated(keep=False)])


 --- df1 --- 

  key  value1
0   A      10
1   B      20

 --- df2 --- 

  key type  value2
0   A    x     100
1   A    y     150
2   B    z     200

 --- (pd.merge(df1, df2, on='key') key 중복 시 행이 늘어남 --- 

  key  value1 type  value2
0   A      10    x     100
1   A      10    y     150
2   B      20    z     200

 --- 중복 key 확인 --- 

  key type  value2
0   A    x     100
1   A    y     150


In [ ]:
# merge: 기준 컬럼 외에 같은 컬럼명이 있으면 suffix가 붙음
df1 = pd.DataFrame({
    "key": ["A", "B"],
    "score": [90, 80]
})

df2 = pd.DataFrame({
    "key": ["A", "B"],
    "score": [95, 85]
})

print("\n --- df1 --- \n")
print(df1)
print("\n --- df2 --- \n")
print(df2)

print("\n --- pd.merge(df1, df2, on='key') 기본 suffix (_x, _y) --- \n")
print(pd.merge(df1, df2, on="key"))

print("\n --- pd.merge(df1, df2, on='key', suffixes=('_old', '_new')) suffixes 직접 지정 --- \n")
print(pd.merge(df1, df2, on="key", suffixes=("_old", "_new")))


 --- df1 --- 

  key  score
0   A     90
1   B     80

 --- df2 --- 

  key  score
0   A     95
1   B     85

 --- pd.merge(df1, df2, on='key') 기본 suffix (_x, _y) --- 

  key  score_x  score_y
0   A       90       95
1   B       80       85

 --- suffixes 직접 지정 --- 

  key  score_old  score_new
0   A         90         95
1   B         80         85
